In [27]:
import pandas as pd
import numpy as np
import re

In [28]:
phones = pd.read_csv('../agg_data/dns_smartphones.csv')
phones.info()

<class 'pandas.DataFrame'>
RangeIndex: 1376 entries, 0 to 1375
Columns: 159 entries, name to Питание
dtypes: float64(10), int64(1), str(148)
memory usage: 1.7 MB


Удалим колонки, в которых больше 30% пропусков, так как данных мало и искусственное заполнение даже более ,чем 30% пропусков, привеедт к смещению значений.

In [29]:
columns = phones.columns
unusless_columns = [column for column in columns if phones[column].isna().sum() > 0.3*len(phones)]
len(unusless_columns)
phones.drop(columns=unusless_columns, inplace=True)
phones.info()

<class 'pandas.DataFrame'>
RangeIndex: 1376 entries, 0 to 1375
Data columns (total 100 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   name                                            1376 non-null   str    
 1   url                                             1376 non-null   str    
 2   price                                           1376 non-null   int64  
 3   Срок эксплуатации                               1369 non-null   str    
 4   Тип                                             1375 non-null   str    
 5   Модель                                          1375 non-null   str    
 6   Год релиза                                      1375 non-null   float64
 7   Игровой                                         1333 non-null   str    
 8   Цвет задней панели                              1371 non-null   str    
 9   Цвет граней                                     137

/tmp/ipykernel_55846/1394198013.py:4: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=unusless_columns, inplace=True)


Обработка колонок

Можно удалить такие колонки как: тип, модель, Диагональ экрана (дюйм), так как эти колонки имеют пропуски, при этом колонка name содержит всю эту информацию без потерь.    
Поэтому информация будет получена за счет колонки name

In [30]:
phones.drop(columns=['Тип', 'Модель', 'Диагональ экрана (дюйм)'], inplace=True)

/tmp/ipykernel_55846/3582166490.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Тип', 'Модель', 'Диагональ экрана (дюйм)'], inplace=True)


#### name 

In [31]:
def prepr_name(x):
    x = x.lower().strip().strip('"')

    diag, x = x.strip().split('" ')
    diag = float(diag)

    x_list = x.strip().split(' ')

    type = x_list.pop(0).strip()
    color = x_list.pop(-1).strip()

    brand = x_list.pop(0).strip()
    x_list.pop(-1) # удаляем гб

    memory = int(x_list.pop(-1).strip())
    model = ' '.join(x_list)

    return {'diag':diag, 'type':type, 'color':color, 'brand':brand, 'memory':memory, 'model':model}

parsed = phones['name'].apply(prepr_name).apply(pd.Series)
phones = pd.concat([phones.drop(columns=['name']), parsed], axis=1)
phones.info()

<class 'pandas.DataFrame'>
RangeIndex: 1376 entries, 0 to 1375
Columns: 102 entries, url to model
dtypes: float64(7), int64(2), str(93)
memory usage: 1.1 MB


#### url

если брать url как id, дубликатов не имеется, сам url больше не пригодится, можно удалить колонку.

In [32]:
print(phones['url'].duplicated().sum())
phones.drop(columns=['url'], inplace=True)

0


/tmp/ipykernel_55846/1888760669.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['url'], inplace=True)


Вспомогательные функции

In [33]:
UNK = 'Unknown'

def num(s, pat=r'(\d+(?:\.\d+)?)'):
    # извлекает числа из строки
    return pd.to_numeric(s.str.extract(pat)[0], errors='coerce')

def has(s, pat):
    # 1/0 по наличию подстроки; где исходный текст пропущен - NaN
    return s.str.contains(pat, case=False, regex=True).astype(float).where(s.notna())

def list_max(s, pat):
    # максимум из всех чисел, найденных по pat ('50+8 Мп' -> 50); пропуск остаётся
    def f(v):
        if pd.isna(v):
            return np.nan
        found = re.findall(pat, v)
        return max(map(float, found)) if found else np.nan
    return s.apply(f)

#### Бинарные колонки

Есть - 1, нет - 0. У nfc опционально считаем 0, так как в базовой версии модуля нет.

In [34]:
FLAGS = [] 
YES_NO = {'есть': 1, 'нет': 0, 'опционально': 0}

binary_cols = {
    'Игровой': 'is_gaming',
    'Поддержка 4G LTE-Advanced': 'lte_advanced',
    'Поддержка VoLTE': 'volte',
    'Укрепленный корпус': 'rugged',
    'Безрамочный экран': 'bezelless',
    'Изогнутый экран': 'curved_screen',
    'Предустановленный пакет Google Mobile Services': 'gms',
    'Стереодинамики': 'stereo_speakers',
    'FM радио': 'fm_radio',
    'NFC': 'nfc',
    'Поддержка OTG': 'otg',
    'Быстрая зарядка': 'fast_charge',
    'Поддержка беспроводной зарядки': 'wireless_charge',
    'Зарядное устройство в комплекте': 'charger_in_box',
    'Двойная фронтальная камера': 'dual_front_cam',
    'Автофокусировка': 'front_autofocus',
    'Встроенная вспышка': 'front_flash',
    'Автофокусировка основной камеры': 'main_autofocus',
    'Оптическая стабилизация': 'ois',
    'Замедленная видеосъемка': 'slow_motion',
    'ИК-порт': 'ir_port',
    'Слот для карты памяти': 'memory_card_slot'
}

for old, new in binary_cols.items():
    phones[old] = phones[old].apply(lambda x: x.lower().strip().split(' ')[0] if (not pd.isna(x)) else x)
    phones[old] = phones[old].apply(lambda x: 'есть' if (x not in YES_NO) and (not pd.isna(x)) else x)
    phones[new] = phones[old].map(YES_NO)

phones = phones.drop(columns=list(binary_cols))

FLAGS += list(binary_cols.values())

#### Экран

- Разрешение экрана -> screen_w, screen_h (меньшая и большая стороны)
- Плотность пикселей удалена, так как вычисляется из разрешения экрана и диагонали (+ меньше пропусков)
- колонка Соотношение сторон была удалена, так как соотношения стороны можно вычислить из разрешения экрана
- Яркость, Частота обновления, Количество цветов экрана -> color_depth_bits, числа и глубина цвета в битах
- Тип матрицы -> panel_type (был укрупнен)

In [35]:
res = phones['Разрешение экрана'].str.extract(r'(\d+)x(\d+)').astype(float)

phones['screen_w'] = res.min(axis=1)
phones['screen_h'] = res.max(axis=1)

phones.drop(columns=['Разрешение экрана', 'Соотношение сторон','Плотность пикселей'], inplace=True)

/tmp/ipykernel_55846/3664743662.py:6: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Разрешение экрана', 'Соотношение сторон','Плотность пикселей'], inplace=True)


извлекли из кол-ва цветов именно глубину цвета, так как по своей сути
это цифра дает информацию о том, сколько градаций яркости может вывести экран
на каждый из 3х каналов
а число цветов получается возведением этого кол-ва в куб

Такое преобразование позволяет моделям, чувствительным к масштабу, делать предсказания лучшего качества.


In [36]:
phones['brightness_nits'] = num(phones['Яркость'])
phones['refresh_hz'] = num(phones['Частота обновления экрана'])

colors = phones['Количество цветов экрана']
mult = colors.str.contains('млрд').map({True: 1e9, False: 1e6})
colors_abs = num(colors) * mult
phones['color_depth_bits'] = np.log2(colors_abs).round()

phones.drop(columns=['Яркость', 'Частота обновления экрана', 'Количество цветов экрана'], inplace=True)

/tmp/ipykernel_55846/3629158017.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Яркость', 'Частота обновления экрана', 'Количество цветов экрана'], inplace=True)


делим экран смартфона на 2 большие группы: oled (retina (фирменный экран apple) также относится к этой группе, так как все модели здесь современные), lcd 

In [37]:
phones['Тип матрицы (подробно)'] = phones['Тип матрицы (подробно)'].str.lower().str.strip()

phones['panel_type'] = 'lcd'

is_oled = phones['Тип матрицы (подробно)'].str.contains('oled|retina', case=False, na=False)
phones.loc[is_oled, 'panel_type'] = 'oled'

phones.loc[phones['Тип матрицы (подробно)'].isna(), 'panel_type'] = np.nan
phones.drop(columns=['Тип матрицы (подробно)'], inplace=True)

phones['notch_type'] = phones['Тип выреза на экране']
phones.drop(columns=['Тип выреза на экране'], inplace=True)

/tmp/ipykernel_55846/3828884110.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Тип матрицы (подробно)'], inplace=True)
/tmp/ipykernel_55846/3828884110.py:12: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Тип выреза на экране'], inplace=True)


In [38]:
INT_COLS = []
INT_COLS += ['refresh_hz', 'color_depth_bits', 'screen_w', 'screen_h']

#### Корпус и цвет

- Тип корпуса -> form_factor
- Материал задней панели -> back_material
- Материал граней -> frame_material
- Защитное покрытие экрана -> screen_glass (семейство стекла (gorilla victus, ceramic shield, kunlun, ...))
- Степень защиты IP → два числа: ip_dust (первая цифра) и ip_water (вторая, максимум из перечисленных). Преобразовываем, чтобы уменьшить число категориальных значений после кодирования в будущем.
- color уже получен из name, три колонки с цветом корпуса/граней/«заявленным производителем» его дублируют — удаляем
- type из name — константа «смартфон», удаляем

In [39]:
phones['form_factor'] = phones['Тип корпуса']
phones.drop(columns=['Тип корпуса'], inplace=True)

phones['back_material'] = phones['Материал задней панели'].str.lower().replace({'стеклопластик': 'стекловолокно'})
phones['frame_material'] = phones['Материал граней'].str.lower()


def glass_family(x):
    if pd.isna(x):
        return np.nan
    x = x.lower()
    for pat, name in [('ceramic shield', 'ceramic shield'), ('gorilla armor', 'gorilla armor'),
                      ('victus', 'gorilla victus'), ('gorilla', 'gorilla glass'),
                      ('kunlun', 'kunlun'), ('panda', 'panda'),
                      ('dragontrail|dt star|agc|asahi', 'dragontrail'), ('schott|xensation', 'schott')]:
        if re.search(pat, x):
            return name
    return 'другое'


phones['screen_glass'] = phones['Защитное покрытие экрана'].apply(glass_family)


def parse_ip(x):
    if pd.isna(x):
        return pd.Series([np.nan, np.nan])
    pairs = re.findall(r'IP([\dX])([\dX])', x)
    dust = [0 if d == 'X' else int(d) for d, w in pairs]
    water = [0 if w == 'X' else int(w) for d, w in pairs]
    return pd.Series([max(dust), max(water)])


phones[['ip_dust', 'ip_water']] = phones['Степень защиты IP'].apply(parse_ip)
INT_COLS += ['ip_dust', 'ip_water']

phones = phones.drop(columns=['Материал задней панели', 'Материал граней', 'Защитное покрытие экрана',
                              'Степень защиты IP', 'Цвет задней панели', 'Цвет граней',
                              'Цвет, заявленный производителем', 'type'])

/tmp/ipykernel_55846/1926157835.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Тип корпуса'], inplace=True)


#### Процессор, графика и память

- Модель процессора -> cpu_model (приведён регистр, Kirin 9010S HiSilicon Kirin 9010S), cpu_vendo и cpu_family (линейка/класс: snapdragon 8/7/6/4, dimensity 9/8/7/6, kirin, exynos 2xxx/1xxx...) — грубый, но информативный «уровень» чипа
- Количество ядер -> cpu_cores
- Техпроцесс -> process_nm
- Максимальная частота, Конфигурация процессора -> cpu_ghz: если не указана, берём максимальную частоту из Конфигурации процессора
- Графический ускоритель -> gpu_vendor
- Тип оперативной памяти -> ram_type (lpddr5x ultra и lpddr5t → lpddr5x);
- Объем встроенной памяти дублирует memory, удаляем
- Объем оперативной памяти -> ram_gb

In [40]:
phones['cpu_model'] = (phones['Модель процессора'].str.lower().str.strip()
                      .str.replace('hisilicon ', '', regex=False).str.replace(r'\s+', ' ', regex=True))

def cpu_vendor(x):
    if pd.isna(x):
        return np.nan
    for pat, name in [('apple', 'apple'), ('snapdragon|qualcomm', 'qualcomm'), ('mediatek|dimensity|helio|mt\\d', 'mediatek'),
                      ('kirin', 'hisilicon'), ('exynos', 'samsung'), ('tensor', 'google'), ('unisoc', 'unisoc')]:
        if re.search(pat, x):
            return name
    return 'другое'


def cpu_family(x):
    if pd.isna(x):
        return np.nan
    for pat, tpl in [(r'snapdragon (\d)', 'snapdragon {}'), (r'dimensity (\d)', 'dimensity {}'),
                     (r'kirin (\d)', 'kirin {}'), (r'exynos (\d)', 'exynos {}xxx')]:
        m = re.search(pat, x)
        if m:
            return tpl.format(m.group(1))
    if 'helio' in x: return 'helio'
    if 'tensor' in x: return 'tensor'
    if 'unisoc' in x: return 'unisoc'
    if 'apple' in x: return 'apple a pro' if 'pro' in x else 'apple a'
    return 'другое'


phones['cpu_vendor'] = phones['cpu_model'].apply(cpu_vendor)
phones['cpu_family'] = phones['cpu_model'].apply(cpu_family)

phones['cpu_cores'] = phones['Количество ядер']
phones['process_nm'] = num(phones['Техпроцесс'])

# если максимальная частота не указана, берём максимум из конфигурации ядер
freq = num(phones['Максимальная частота процессора'])
freq_cfg = list_max(phones['Конфигурация процессора'], r'(\d+(?:\.\d+)?)\s*ГГц')
phones['cpu_ghz'] = freq.fillna(freq_cfg).round(2)

phones['gpu'] = phones['Графический ускоритель'].str.lower().str.strip()


def gpu_vendor(x):
    if pd.isna(x):
        return np.nan
    for pat, name in [('adreno', 'adreno'), ('mali|immortalis', 'arm mali'), ('apple', 'apple'),
                      ('xclipse', 'xclipse'), ('maleoon', 'maleoon'), ('powervr|img', 'powervr')]:
        if re.search(pat, x):
            return name
    return 'другое'


phones['gpu_vendor'] = phones['gpu'].apply(gpu_vendor)

ram_type = phones['Тип оперативной памяти'].str.lower().str.strip()
phones['ram_type'] = ram_type.str.replace(r'^lpddr5(x|t).*', 'lpddr5x', regex=True)
phones['ram_gb'] = num(phones['Объем оперативной памяти'])

INT_COLS += ['cpu_cores', 'process_nm', 'ram_gb']
phones = phones.drop(columns=['Модель процессора', 'Количество ядер', 'Техпроцесс', 'Максимальная частота процессора',
                              'Конфигурация процессора', 'Графический ускоритель', 'Тип оперативной памяти',
                              'Объем оперативной памяти', 'Объем встроенной памяти', 'gpu'])

#### ОС

* Операционная система -> os (android/ios; у бренда apple пропуск однозначно ios)
* Версия ОС -> os_version

In [41]:
phones['os'] = phones['Операционная система'].str.lower()
phones.loc[phones['os'].isna() & (phones['brand'] == 'apple'), 'os'] = 'ios'   # расчёт
phones['os_version'] = num(phones['Версия ОС'])
INT_COLS += ['os_version']
phones = phones.drop(columns=['Операционная система', 'Версия ОС'])

#### Камеры

- Тип модулей камеры -> флаги has_telephoto, has_periscope, has_ultrawide, has_macro
- Формат видеосъемки -> video_max_res (порядковая шкала: HD=1, Full HD=2, Quad HD=3, 4K=4, 8K=5);
- Тип вспышки -> flash_type (тройная и четверная объединены)
- Цифровой зум (фото) -> digital_zoom
- Количество мегапикселей основной камеры, Апертура основной камеры, Апертура фронтальной камеры, Количество мегапикселей фронтальной камеры - удалены, так как из-за формата и кол-ва информации невозможно сопоставить данные о мп и апертуре для каждой камеры (их несколько: и основных, и иногда фронтальных). Сами по себе мп не несут информации о качестве камеры. Необходимы оба признака для ее характеристики. Хоть апертура и является важной характеристикой, не для всего кол-ва камер она указана, то есть общее число камер (фронтальных или основных) не совпадает с кол-вом значений апертуры.

In [42]:
mp = phones['Количество мегапикселей основной камеры']
n_mp = mp.apply(lambda v: len(re.findall(r'\d+(?:\.\d+)?', v)) if pd.notna(v) else np.nan)
phones['rear_cams'] = phones['Количество основных (тыловых) камер'].fillna(n_mp)

mod = phones['Тип модулей камеры'].str.lower()
for name, pat in [('has_telephoto', 'телефото'), ('has_periscope', 'перископ'),
                  ('has_ultrawide', 'сверхширок|ультраширок'), ('has_macro', 'макро')]:
    phones[name] = has(mod, pat)
    FLAGS.append(name)

phones['digital_zoom'] = num(phones['Цифровой зум (фото)'])

VIDEO_ORDER = {'hd': 1, 'full hd': 2, 'quad hd': 3, 'ultra hd 4k': 4, 'ultra hd 8k': 5}

def video_max(x):
    if pd.isna(x):
        return np.nan
    
    # Оставляем только известные ключи в нижнем регистре
    valid_items = [t.strip().lower() for t in str(x).split(',') if t.strip().lower() in VIDEO_ORDER]
    
    if not valid_items:
        return np.nan
        
    # Сравниваем элементы по их рангу из словаря, но возвращаем саму строку
    return max(valid_items, key=VIDEO_ORDER.get)


phones['video_max_res'] = phones['Формат видеосъемки'].apply(video_max)

flash = phones['Тип вспышки'].str.split().str[0]
phones['flash_type'] = flash.replace({'тройная': 'три и более', 'четверная': 'три и более'})

INT_COLS += ['rear_cams']
phones = phones.drop(columns=['Количество мегапикселей основной камеры', 'Количество основных (тыловых) камер',
                              'Тип модулей камеры', 'Апертура основной камеры', 'Апертура фронтальной камеры',
                              'Количество мегапикселей фронтальной камеры', 'Цифровой зум (фото)',
                              'Формат видеосъемки', 'Тип вспышки'])

#### Связь и интерфейс

- Поддержка стандартов сотовой связи -> has_5g
- Количество физических SIM-карт, Количество eSIM -> sim_physical, esim_count (если число eSIM не указано, но в Формате SIM-карт есть eSIM, то 1 — расчёт);
- Версия Bluetooth -> bluetooth_ver
- Стандарт Wi-Fi -> wifi_gen (максимум из перечисленных: n=4, ac=5, ax=6, be=7, так как максимум подразумевает совместимость с прошлыми версиями и )
- Системы навигации -> n_nav_systems (сколько систем навигации перечислено)
- Разъем для наушников -> headphone_jack - есть ли специальный разъем (круглый)
- Датчики удалены, так как список свернут
- диапазоны 2G/3G/4G — удаляем: 2G почти константа, 3G — региональные наборы, а 4G в парсинге обрезан в 90% строк
- Интерфейс для зарядки (Lightning следует из бренда Apple) и Формат SIM-карт — удаляем.

In [43]:
phones['has_5g'] = has(phones['Поддержка стандартов сотовой связи'], '5G')

phones['sim_physical'] = phones['Количество физических SIM-карт']
phones['esim_count'] = phones['Количество eSIM'].fillna(has(phones['Формат SIM-карт'], 'esim'))

phones['bluetooth_ver'] = phones['Версия Bluetooth']
WIFI = {'n': 4, 'ac': 5, 'ax': 6, 'be': 7}
phones['wifi_gen'] = phones['Стандарт Wi-Fi'].apply(
    lambda v: max([WIFI[t] for t in re.findall(r'802\.11(\w+)', v) if t in WIFI], default=np.nan) if pd.notna(v) else np.nan)
phones['n_nav_systems'] = phones['Системы навигации'].apply(lambda v: len(v.split(',')) if pd.notna(v) else np.nan)

phones['headphone_jack'] = has(phones['Разъем для наушников'], 'jack')
FLAGS += ['has_5g', 'headphone_jack']
INT_COLS += ['sim_physical', 'esim_count', 'wifi_gen', 'n_nav_systems']

phones = phones.drop(columns=['Поддержка стандартов сотовой связи', 'Диапазоны частот 2G', 'Диапазоны частот 3G',
                              'Диапазоны частот 4G (LTE)', 'Формат SIM-карт', 'Количество физических SIM-карт',
                              'Количество eSIM', 'Версия Bluetooth', 'Стандарт Wi-Fi', 'Системы навигации',
                              'Интерфейс для зарядки', 'Разъем для наушников', 'Датчики'])

#### Аккумулятор и зарядка

- Емкость аккумулятора -> battery_mah
- Тип аккумулятора -> battery_type
- Выходная мощность ЗУ -> charger_in_box (флаг 1/0 - есть ли зарядка в комплекте или нет), charger_w_in_box (мощность устройства, если оно есть в комплекте, 0 - если нет)
- Требуемая мощность зарядного устройства (25Вт беспроводное ЗУ, 40Вт сетевое ЗУ) -> wired_charge_w и wireless_charge_w — необходимая мощность для каждого типа зарядки
- Поддержки реверсивной зарядки -> reverse_charge / reverse_wireless
- Напряжение питания ЗУ дублирует charger_in_box, удаляем

In [44]:
phones['battery_mah'] = num(phones['Емкость аккумулятора'])

phones['battery_type'] = phones['Тип аккумулятора'].str.lower()

cw = num(phones['Выходная мощность ЗУ'])
phones['charger_in_box'] = phones['charger_in_box'].fillna((cw > 0).astype(float).where(cw.notna()))
phones['charger_w_in_box'] = cw.where(~(phones['charger_in_box'] == 0), 0)

req = phones['Требуемая мощность зарядного устройства']
wired = list_max(req, r'(\d+(?:\.\d+)?)\s*Вт\s*сетевое')
wireless = list_max(req, r'(\d+(?:\.\d+)?)\s*Вт\s*беспроводное')
wired = wired.where(req.isna(), wired.fillna(0))
wireless = wireless.where(req.isna(), wireless.fillna(0))
wireless = wireless.fillna(phones['wireless_charge'].map({0: 0.0}))       # расчёт: флаг=0 -> беспроводной зарядки нет
phones['wireless_charge'] = phones['wireless_charge'].fillna((wireless > 0).astype(float).where(wireless.notna()))
phones['wired_charge_w'] = wired
phones['wireless_charge_w'] = wireless

rev = phones['Поддержка реверсивной зарядки'].str.lower()
phones['reverse_charge'] = (~rev.str.contains('нет')).astype(float).where(rev.notna())
phones['reverse_wireless'] = has(rev, 'беспровод')
FLAGS += ['reverse_charge', 'reverse_wireless']

phones = phones.drop(columns=['Емкость аккумулятора', 'Тип аккумулятора', 'Напряжение питания ЗУ', 'Выходная мощность ЗУ',
                              'Требуемая мощность зарядного устройства', 'Поддержка реверсивной зарядки'])

In [45]:
phones['charger_w_in_box'].unique()

array([  0. ,  80. ,  45. ,  66. ,  35. ,  15. , 100. , 120. ,  22.5,
        40. ,  10. ,  33. ,  90. ,  67. ,  60. ,   nan,  70. ,  30. ,
        68. , 125. ,  36. ,  18. ,  44. ])

#### Биометрия
- Биометрическая защита -> face_unlock, fingerprint;

In [46]:
bio = phones['Биометрическая защита']
phones['face_unlock'] = has(bio, 'лица')
phones['fingerprint'] = has(bio, 'отпечатк')
FLAGS += ['face_unlock', 'fingerprint']

#### Размеры, вес    
- Ширина, Высота, Толщина (мм), Вес (г) -> width_mm, height_mm, thickness_mm, weight_g

In [47]:
for new, old in [('width_mm', 'Ширина'), ('height_mm', 'Высота'), ('thickness_mm', 'Толщина'), ('weight_g', 'Вес смартфона')]:
    phones[new] = num(phones[old])

#### Cрок эксплуатации -> useful_life 

In [48]:
def prepr_useful_life(x):
    if not pd.isna(x):
        x = int(x.strip().split(' ')[0])
    return x


phones['useful_life'] = phones['Срок эксплуатации'].apply(prepr_useful_life)
phones.drop(columns=['Срок эксплуатации'], inplace=True)

/tmp/ipykernel_55846/4290840648.py:8: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Срок эксплуатации'], inplace=True)


#### Год релиза -> year_release

In [49]:
phones['year_release'] = phones['Год релиза']
phones.drop(columns=['Год релиза'], inplace=True)

/tmp/ipykernel_55846/772466471.py:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  phones.drop(columns=['Год релиза'], inplace=True)


#### Прочее

- удаляются: Комплектация (свёрнутый список; зарядка в комплекте уже есть как charger_in_box), Режимы и функции фотосъемки/съемки (объемные и свернутые списки, которые не повзоляют даже посчитать реальное кол-во режимов), Особенности и функции видеосъемки (-//-), Разрешение видео и частота кадров (-//-), Разрешение видеосъемки (-//-), Форматы аудиофайлов, Форматы воспроизведения видео, Прочие технологии передачи данных (длинные свободные списки, 70-240 уникальных значений), scraped_at (время парсинга).

In [50]:
phones = phones.drop(columns=['Биометрическая защита', 'Ширина', 'Высота', 'Толщина', 'Вес смартфона',
                              'Комплектация', 'Режимы и функции фотосъемки', 'Режимы и функции съемки',
                              'Особенности и функции видеосъемки', 'Разрешение видео и частота кадров',
                              'Разрешение видеосъемки', 'Форматы аудиофайлов', 'Форматы воспроизведения видео',
                              'Прочие технологии передачи данных', 'scraped_at'], errors='ignore')

In [51]:
phones.info()

<class 'pandas.DataFrame'>
RangeIndex: 1376 entries, 0 to 1375
Data columns (total 82 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              1376 non-null   int64  
 1   diag               1376 non-null   float64
 2   color              1376 non-null   str    
 3   brand              1376 non-null   str    
 4   memory             1376 non-null   int64  
 5   model              1376 non-null   str    
 6   is_gaming          1333 non-null   float64
 7   lte_advanced       1195 non-null   float64
 8   volte              1360 non-null   float64
 9   rugged             1363 non-null   float64
 10  bezelless          1362 non-null   float64
 11  curved_screen      1370 non-null   float64
 12  gms                1300 non-null   float64
 13  stereo_speakers    1272 non-null   float64
 14  fm_radio           1092 non-null   float64
 15  nfc                1330 non-null   float64
 16  otg                1336 non-null   

In [52]:
phones.to_csv('dns_smartphones_cleaned.csv', index_label='id')